In [ ]:
%matplotlib widget 
%load_ext autoreload
import os
import time
thisfiledir=os.path.abspath("")
deepracingrepodir=os.path.normpath(os.path.join(thisfiledir, "..", ".."))
import sys
sys.path = [os.path.join(deepracingrepodir, "deepracing_py"), os.path.join(deepracingrepodir, "DCNN-Pytorch"), thisfiledir] + sys.path
import matplotlib.transforms
import deepracing, deepracing_models.math_utils as mu, deepracing.path_utils
from deepracing_models.math_utils.bounds_checking import BoundsChecker
from deepracing_models.math_utils.statistics import CollisionProbabilityEstimator
from deepracing_models.probabilistic_models import ProbabilisticBezierCurve
import deepracing_models.data_loading.file_datasets as FD
import torch, numpy as np
from scipy.spatial.transform import Rotation, RotationSpline
import matplotlib.figure, matplotlib.axes, matplotlib.collections, matplotlib.patches
from matplotlib import pyplot as plt
import torch.distributions
import torch.utils.data as torchdata
import scipy.interpolate, scipy.spatial
from tqdm import tqdm
import PIL, PIL.Image, PIL.ImageOps
import utils
import yaml
searchdirs = []
try:
    searchdirs.extend(os.environ["F1_MAP_DIRS"].split(os.pathsep))
except ImportError as e:
    pass
try:
    import ament_index_python # type: ignore
    searchdirs.append(os.path.join(ament_index_python.get_package_share_directory("deepracing_launch"), "maps"))
except ImportError as e:
    pass
except ament_index_python.packages.PackageNotFoundError as e:
    pass
transform_to_map=True

datadir = "/p/DeepRacing/overtaking_datasets" #/Jeddah_2023_7_13_15_27"
dsets = []
for subdir in os.listdir(datadir):
    # print(subdir)
    # with 
    with open(os.path.join(datadir, subdir,"metadata.yaml"),"r") as f:
        metadata = yaml.load(f, Loader=yaml.SafeLoader)
    with open(os.path.join(datadir, subdir,"data.npz"),"rb") as f:
        data = np.load(f)
        dsets.append(FD.OvertakingTrajectoriesDataset(data, metadata))
# delta_t = datadict["delta_t"]
concatdset = torchdata.ConcatDataset(dsets)
idx_rand = 64
# idx_rand = 138
# idx_rand = int(np.random.randint(0, high=len(concatdset), size=1))
datadict = concatdset[idx_rand]
tfit = torch.as_tensor(datadict["delta_t"], dtype=torch.float64)
tfit = tfit-tfit[0]
attacker_positions = torch.as_tensor(datadict["attacker_pos"]).type_as(tfit)
defender_positions = torch.as_tensor(datadict["defender_pos"]).type_as(tfit)
attacker_quats = torch.as_tensor(datadict["attacker_quat"]).type_as(tfit)
defender_quats = torch.as_tensor(datadict["defender_quat"]).type_as(tfit)
# # delta_t = delta_t - delta_t[0]
trackmap = deepracing.searchForTrackmap(datadict["track_name"], searchdirs, align=True, transform_to_map=transform_to_map)
Nparticles = int(round(1.25*(2**9)))
print(datadict.keys())
plotsdir=os.path.join(os.environ["HOME"], "plots")

In [ ]:

car_width = 2.0
car_length = 4.5
print("Building centerline helper")
centerline_structured : np.ndarray = trackmap.centerline
position_keys =["x", "y", "z"]
quaternion_keys = ["i", "j", "k", "w"]
width_map = trackmap.width_map

shrink_factor = 1.0
left_widths = shrink_factor*torch.as_tensor(width_map["ob_distance"]).type_as(tfit).squeeze(-1) + 0.25*car_width
right_widths = shrink_factor*torch.as_tensor(width_map["ib_distance"]).type_as(tfit).squeeze(-1) - 0.25*car_width

centerline_dense_full = torch.as_tensor(np.concatenate([width_map[k] for k in position_keys], axis=1)).type_as(left_widths)
centerline_dense = centerline_dense_full[...,[0,1]]
centerline_quats = np.concatenate([width_map[k] for k in quaternion_keys], axis=1)
centerline_rots : Rotation = Rotation.from_quat(centerline_quats/np.linalg.norm(centerline_quats, ord=2.0, axis=-1, keepdims=True))
centerline_rotmats : torch.Tensor = torch.as_tensor(centerline_rots.as_matrix(), device=centerline_dense.device, dtype=centerline_dense.dtype)
ib_dense_full = (centerline_dense_full + centerline_rotmats[...,1]*right_widths[:,None]).cpu()
ib_dense = ib_dense_full[...,[0,1]]
ob_dense_full = (centerline_dense_full + centerline_rotmats[...,1]*left_widths[:,None]).cpu()
ob_dense = ob_dense_full[...,[0,1]]

left_widths = torch.norm(ob_dense - centerline_dense, p=2.0, dim=-1)
right_widths = -torch.norm(ib_dense - centerline_dense, p=2.0, dim=-1)
dT_desired = tfit[-1].item()
print("Built centerline helper")

print("Building boundary helpers")
innerbound_helper = mu.SimplePathHelper.from_closed_path(ib_dense, 0.5).to(dtype=left_widths.dtype, device=left_widths.device)
outerbound_helper = mu.SimplePathHelper.from_closed_path(ob_dense, 0.5).to(dtype=left_widths.dtype, device=left_widths.device)
print("Built boundary helpers")


# print("Building Raceline helper")

# raceline_helper = mu.RacelineHelper.from_closed_path()
# print("Built Raceline helper")

In [ ]:

# idx0 = int(np.random.randint(0, high=int(round(0.9*line_all_times.shape[0]))))
# t0 = float(line_all_times[idx0].item())
# t0 = 25.330
# t0 = 55.591
# t0 = 8.25
# t0 = 75.16829681396484
# t0 = 46.25149917602539
# t0 = 46.551
# t0 = 52.30149841308594 + 3.0
# print("t0:", t0)
bounds_checker : BoundsChecker = (
    BoundsChecker(gauss_order=20, dT=dT_desired, stdev=0.875,
        refline_points=centerline_dense, dr_samp=0.1, left_widths=left_widths, right_widths=right_widths).eval()
).to(dtype=left_widths.dtype, device=left_widths.device)
bounds_checker.rebuild_kdtree()



# Pfit = defender_positions[:,:2]
# Pfit_target = attacker_positions[:,:2]
target_skip=4
Pfit = attacker_positions[:,:2]
Pfit_target = defender_positions[target_skip:-target_skip,:2]

Q0 = attacker_quats[0].clone()
R0 = Rotation.from_quat(Q0.cpu().numpy())
Qmask = torch.as_tensor([0.0, 0.0, 1.0, 1.0]).type_as(attacker_quats)

Qfit = attacker_quats*Qmask[None]
Qfit_target = defender_quats*Qmask[None]

egovehicle_rotspline = RotationSpline(tfit.cpu().numpy(), Rotation.from_quat(Qfit.cpu().numpy()))
targetvehicle_rotspline = RotationSpline(tfit.cpu().numpy(), Rotation.from_quat(Qfit_target.cpu().numpy()))

Targetvehicle_curve, Targetvehicle_tswitch = mu.compositeBezierFit(torch.linspace(0.0, dT_desired, steps=Pfit_target.shape[0]).type_as(Pfit_target)[None], Pfit_target[None], 6, 
                                                            #   dYdT_0=V0[None],
                                                              Y_0=Pfit_target[[0,]],
                                                              Y_f=Pfit_target[[-1,]],
                                                              kbezier=3,
                                                              constraint_level=2)
Targetvehicle_curve = Targetvehicle_curve[0]
Targetvehicle_tstart = Targetvehicle_tswitch[0,:-1]
Targetvehicle_dT = torch.diff(Targetvehicle_tswitch[0], dim=0)
Targetvehicle_curve_deriv = (Targetvehicle_curve.shape[-2]-1)*torch.diff(Targetvehicle_curve, dim=-2)/Targetvehicle_dT[:,None,None]

Curveparticles_mean, Curvefit_tswitch = mu.compositeBezierFit(tfit[None], Pfit[None], Targetvehicle_curve.shape[-3], 
                                                            #   dYdT_0=V0[None],
                                                              Y_0=Pfit[[0,]],
                                                              Y_f=Pfit[[-1,]],
                                                              kbezier=Targetvehicle_curve.shape[-2]-1,
                                                              constraint_level=2)


Curveparticle_tstart = Curvefit_tswitch[:,:-1].expand(Nparticles, Curvefit_tswitch.shape[-1]-1).clone()
Curveparticle_dT = torch.diff(Curvefit_tswitch, dim=1).expand(Nparticles, Curvefit_tswitch.shape[-1]-1).clone()
Curveparticles = Curveparticles_mean.expand([Nparticles,] + list(Curveparticles_mean.shape[1:])).clone()
Curveparticles_mean_deriv = (Curveparticles_mean.shape[-2]-1)*torch.diff(Curveparticles_mean, dim=-2)/Curveparticle_dT[[0,],...,None,None]
image_scale = 1.0#5*shrink_factor
lat_buffer = 0.5*image_scale*car_width
long_buffer = 0.5*image_scale*car_length
collision_probability_estimator : CollisionProbabilityEstimator = CollisionProbabilityEstimator(
    12, dT_desired, 16, lat_buffer, long_buffer
    ).to(dtype=tfit.dtype, device=tfit.device)
gauss2d_integrator = collision_probability_estimator.gaussian_pdf_integrator


In [ ]:

p0 : torch.Tensor = Curveparticles_mean[0,0,0].clone()
pf : torch.Tensor = Curveparticles_mean[0,-1,-1].clone() 
((cl_r0, cl_rf), (cl_p0, cl_pf), (cl_tau_0, cl_tau_f), (cl_nu_0, cl_nu_f), _) = bounds_checker.refline_helper.closest_point_approximate(torch.stack([p0, pf]), newton_iterations=5)

cl_R0 = torch.stack([cl_tau_0, cl_nu_0], dim=1)
cl_Rf = torch.stack([cl_tau_f, cl_nu_f], dim=1)

ib_r0, ib_rf = innerbound_helper.y_axis_intersection(torch.stack([cl_p0, cl_pf]), torch.stack([cl_R0, cl_Rf], dim=0))
ob_r0, ob_rf = outerbound_helper.y_axis_intersection(torch.stack([cl_p0, cl_pf]), torch.stack([cl_R0, cl_Rf], dim=0))

asdf = innerbound_helper(torch.linspace(ib_r0, ib_rf, steps=300).type_as(ib_r0))
ib_plot : torch.Tensor = asdf[0].clone()
asdf = outerbound_helper(torch.linspace(ob_r0, ob_rf, steps=ib_plot.shape[0]).type_as(ob_r0))
ob_plot : torch.Tensor = asdf[0].clone()




In [ ]:
import matplotlib.colors, matplotlib.patches
figname="Overtake Slice"
columnwidth=3.5
figsize=1.0*np.asarray([columnwidth, columnwidth])
fig, ax = plt.subplots(label=figname, figsize=figsize, layout="tight", frameon=False, clear=True)
# plt.figure()

carimage : PIL.Image.Image = PIL.Image.open(os.path.join(thisfiledir, "assets", "cavalier_transparent.png"))
invertedcarimage : PIL.Image.Image = PIL.Image.open(os.path.join(thisfiledir, "assets", "cavalier_transparent_inverted.png"))

caraspect = car_length/car_width
carimage = carimage.resize((carimage.width, int(round(carimage.width*caraspect))), resample=PIL.Image.LANCZOS)
invertedcarimage = invertedcarimage.resize((invertedcarimage.width, int(invertedcarimage.width*caraspect)), resample=PIL.Image.LANCZOS)
# invertedcarimage : PIL.Image.Image = PIL.ImageOps.invert(carimage.convert('RGB')).convert('RGBA')
# invertedcarimage.putalpha(carimage.getchannel("A"))
# gauss2d_integrator collision_probability_estimator.gl1d.eta

flip = torch.as_tensor([-1.0, 1.0]).type_as(tfit)
tplot = torch.linspace(tfit[0], tfit[-1], steps=120).type_as(tfit)
# timage = collision_probability_estimator.gl1d.eta
timage = tplot
(ego_pplot,), _ = mu.compositeBezierEval(Curveparticle_tstart[[0,]], Curveparticle_dT[[0,]], Curveparticles_mean, tplot[None])
(target_pplot,), _ = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve, tplot[None])

egoplotline, = ax.plot(*(ego_pplot.cpu().T), label="Ego Trajectory", color=utils.COLORS.UVA_ORANGE, linewidth=1)
egoplotline.set_zorder(egoplotline.get_zorder()-1)
targetplotline, = ax.plot(*(target_pplot.cpu().T), label="Target Trajectory", linestyle=egoplotline.get_linestyle(), color=1.0-utils.COLORS.UVA_ORANGE, linewidth=1)
targetplotline.set_zorder(egoplotline.get_zorder())
# ax.xaxis.set_ticks([])
# ax.yaxis.set_ticks([])
# for k in ax.spines.keys():
#     ax.spines[k].set_visible(False)
# ax.set_aspect(aspect=2.0, adjustable="box")
# ax.set_axis_off()
#, bbox_inches="tight"
fig.savefig(os.path.join(plotsdir, "%s.bare.svg" % (figname.lower().replace(" ","_"),)), transparent=True, pad_inches=0.0, bbox_inches="tight")

ibplotline, = ax.plot(*(ib_plot.cpu().T), color="black", label="Track Boundaries")
ibplotline.set_zorder(egoplotline.get_zorder())
obplotline, = ax.plot(*(ob_plot.cpu().T), color=ibplotline.get_color(), linestyle=ibplotline.get_linestyle())
obplotline.set_zorder(egoplotline.get_zorder())

allpoints = torch.cat([ego_pplot[:,:2], target_pplot[:,:2], ob_plot, ib_plot], dim=0)

ego_pimage, idxbuckets = mu.compositeBezierEval(Curveparticle_tstart[[0,]], Curveparticle_dT[[0,]], Curveparticles_mean, timage[None])
ego_pimage = ego_pimage[0]
ego_rotmatimage = torch.as_tensor(egovehicle_rotspline(timage.cpu().numpy()).as_matrix())[:,0:2,0:2].type_as(ego_pimage)
ego_tauimage = ego_rotmatimage[:,:,0]
# ego_vimage, _ = mu.compositeBezierEval(Curveparticle_tstart[[0,]], Curveparticle_dT[[0,]], Curveparticles_mean_deriv, timage[None], idxbuckets=idxbuckets)
# ego_vimage = ego_vimage[0]
# ego_tauimage = ego_vimage/torch.norm(ego_vimage, dim=-1, p=2.0, keepdim=True)
# ego_rotmatimage = torch.stack([ego_tauimage, ego_tauimage[:,[1,0]]*flip[None]], dim=-1)
boxpoints = (torch.stack(torch.meshgrid([
    0.5*torch.as_tensor([-car_length, car_length]),
    0.5*torch.as_tensor([-car_width, car_width])
], indexing="ij"), dim=0).reshape(2,-1).type_as(ego_tauimage))
boxpoints = image_scale*boxpoints[:,torch.argsort(torch.atan2(boxpoints[1], boxpoints[0]))]
# print(boxpoints.T)
target_pimage, idxbuckets = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve[None], timage[None])
target_pimage = target_pimage[0]
# target_rotmatimage = torch.as_tensor(targetvehicle_rotspline(timage.cpu().numpy()).as_matrix())[:,0:2,0:2].type_as(target_pimage)
# target_tauimage = target_rotmatimage[:,:,0]
target_vimage, _ = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve_deriv[None], timage[None])
target_vimage = target_vimage[0]
target_tauimage = target_vimage/torch.norm(target_vimage, dim=-1, p=2.0, keepdim=True)
target_rotmatimage = torch.stack([target_tauimage, target_tauimage[...,[1,0]]*flip[None]], dim=-1)
for idx_image in range(timage.shape[0]):
    # _,_,mplimage1=utils.plot_image(ax, carimage, ego_pimage[idx_image], ego_tauimage[idx_image], car_width, car_length, image_scale=image_scale, aspect="equal")
    # mplimage1.set_zorder(mplimage1.get_zorder()+1)

    boxpoints_ego = ((ego_rotmatimage[idx_image]@boxpoints).T + ego_pimage[idx_image])#[[1,3,2,0]]
    # egopoly : matplotlib.patches.Polygon = ax.add_patch(matplotlib.patches.Polygon(boxpoints_ego.cpu().numpy(), alpha=0.25, closed=True, fill=False, edgecolor=utils.COLORS.UVA_ORANGE, linestyle="solid"))
    # egopoly.set_linewidth(0.85*figsize[0]/columnwidth)

    # _,_,mplimage2=utils.plot_image(ax, invertedcarimage, target_pimage[idx_image], target_tauimage[idx_image], car_width, car_length, image_scale=image_scale, aspect="equal")
    # mplimage2.set_zorder(mplimage1.get_zorder())
    boxpoints_target = ((target_rotmatimage[idx_image]@boxpoints).T + target_pimage[idx_image])#[[1,3,2,0]]
    # targetpoly : matplotlib.patches.Polygon = ax.add_patch(matplotlib.patches.Polygon(boxpoints_target.cpu().numpy(), alpha=egopoly.get_alpha(), closed=True, fill=False, edgecolor=1.0-utils.COLORS.UVA_ORANGE, linestyle="solid"))
    # targetpoly.set_linewidth(egopoly.get_linewidth())
# ax.scatter(*(target_pimage[:,[0,1]].cpu().T), s=2**(2.5), color=1.0-utils.COLORS.UVA_ORANGE)
# ax.scatter(*(ego_pimage[:,[0,1]].cpu().T), s=2**(2.5), color=utils.COLORS.UVA_ORANGE)
# gaussianmeans_01 = (image_scale*torch.stack(torch.meshgrid([
#     0.5*torch.linspace(-car_length, car_length, steps=4),
#     0.5*torch.linspace(-car_width, car_width, steps=3)
# ], indexing="ij"), dim=0)).reshape(2,-1).T.type_as(target_rotmatimage)
ls = torch.linspace(0.0, 1.0, steps=4)
gaussianmeans_01 = torch.cat([boxpoints.T.clone(), torch.zeros_like(boxpoints[:,0])[None]], dim=0)
print(gaussianmeans_01)
gaussianmeans_target = (target_rotmatimage[:,None] @ gaussianmeans_01[None,...,None]).squeeze(-1) + target_pimage[:,None]
#, collision_probs, overall_lambdas, overall_collision_free_probs
logtwopi = float(np.log(2.0*np.pi))
target_stdevs = (0.9*image_scale*torch.as_tensor([1.0, 0.75]).type_as(target_rotmatimage))[None].expand(target_rotmatimage.shape[:-1])
target_logstdevs = (logtwopi + torch.log(target_stdevs).sum(dim=-1))
target_stdev_inv_matrix = (torch.diag_embed(1.0/target_stdevs)@target_rotmatimage.transpose(-2,-1))
gauss_pts, gaussian_pdf_vals, dense_collision_probs = \
    gauss2d_integrator(gaussianmeans_target, target_stdev_inv_matrix, target_logstdevs,
                                    ego_rotmatimage[None], ego_pimage[None])
gauss_pts = gauss_pts[0]
dense_collision_probs = dense_collision_probs[0].clip(0.0, 1.0)
gaussian_pdf_vals=gaussian_pdf_vals[0]
print("gaussianmeans_target.shape:", gaussianmeans_target.shape)
print("target_stdev_inv_matrix.shape:", target_stdev_inv_matrix.shape)
print("target_logstdevs.shape:", target_logstdevs.shape)
print("ego_rotmatimage.shape:", ego_rotmatimage.shape)
print("ego_pimage.shape:", ego_pimage.shape)

minx, maxx = allpoints[:,0].min().item() - 0.5*image_scale*car_length, allpoints[:,0].max().item() + 0.5*image_scale*car_length,
miny, maxy = allpoints[:,1].min().item() - 0.5*image_scale*car_length, allpoints[:,1].max().item() + 0.5*image_scale*car_length,
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_axis_off()
ax.set_aspect(1.0)

tci = torch.linspace(0.0, dT_desired, steps=100).type_as(tfit)
(pci,), _ = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve[None], tci[None])
(vci,), _ = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve_deriv[None], tci[None])
tauci = vci/torch.norm(vci, dim=-1, p=2.0, keepdim=True)
nuci = tauci[:,[1,0]]*flip[None]

pci = torch.cat([pci, pci[[-1,]] + tauci[[-1,]]*0.5*car_length], dim=0)
nuci = torch.cat([nuci, nuci[[-1,]]], dim=0)
spread = torch.linspace(0.05*car_width, 2.25*car_width, steps=tci.shape[0]+1).type_as(tci)
cibound = torch.cat([pci + spread[:,None]*nuci, torch.flip(pci - spread[:,None]*nuci, dims=[0,])[:-1]], dim=0)
cipoly : matplotlib.patches.Polygon = ax.add_patch(matplotlib.patches.Polygon(cibound.cpu().numpy(), closed=True, fill=True, edgecolor=1.0-utils.COLORS.UVA_ORANGE, alpha=0.5, linestyle="dashed"))

fig.savefig(os.path.join(plotsdir, "%s.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)

In [ ]:
# %autoreload 2
# import utils
import matplotlib.patches
import torch.distributions
figname="Overtake Inset"
plt.close(fig=figname)
fig_inset, ax_inset = plt.subplots(label=figname, figsize=figsize, layout="constrained", frameon=False)
visible_index = torch.argmin(torch.abs(timage-dT_desired*0.75))
#torch.argmax(((ego_pimage[:,0]>=xlim_inset[0])*(ego_pimage[:,0]<=xlim_inset[1])*(ego_pimage[:,1]>=ylim_inset[0])*(ego_pimage[:,1]<=ylim_inset[1])).float())
alpha=0.3
Tmat_ego, _, egoim_inset = utils.plot_image(ax_inset, carimage, ego_pimage[visible_index], ego_tauimage[visible_index], car_width, car_length, 
                                  image_scale=image_scale, alpha=alpha)
egoim_inset.set_zorder(1)
Tmat_target, _, targetim_inset = utils.plot_image(ax_inset, invertedcarimage, target_pimage[visible_index], target_tauimage[visible_index], car_width, car_length, 
                                     image_scale=image_scale, alpha=alpha)
targetim_inset.set_zorder(egoim_inset.get_zorder())
bbox_ego = ((Tmat_ego[0:2,0:2] @ (image_scale*torch.stack(torch.meshgrid([
    0.5*torch.as_tensor([-car_length, car_length]),
    0.5*torch.as_tensor([-car_width, car_width])
], indexing="ij"), dim=0).reshape(2,-1).type_as(Tmat_ego))).T + Tmat_ego[0:2,2])[[0,1,3,2]]
bbox_target = ((Tmat_target[0:2,0:2] @ (image_scale*torch.stack(torch.meshgrid([
    0.5*torch.as_tensor([-car_length, car_length]),
    0.5*torch.as_tensor([-car_width, car_width])
], indexing="ij"), dim=0).reshape(2,-1).type_as(Tmat_target))).T + Tmat_target[0:2,2])[[0,1,3,2]]
integration_vals = dense_collision_probs[visible_index]
ellipse_alphas = ((integration_vals/torch.max(integration_vals))**0.25).clip(0.2, 1.0)
angle = torch.atan2(Tmat_target[1,0], Tmat_target[0,0]).item()
num_ellipses=25
ellipse_patches = utils.plot_gaussian(ax_inset, gaussianmeans_target[visible_index,0], target_stdevs[visible_index], angle, num_ellipses=num_ellipses, alpha=ellipse_alphas[0].item())
for i in range(1, gaussianmeans_target.shape[1]):
    ellipse_patches.extend(utils.plot_gaussian(ax_inset, gaussianmeans_target[visible_index,i], target_stdevs[visible_index], angle, num_ellipses=num_ellipses, alpha=ellipse_alphas[i].item()))

bbox_ego_patch : matplotlib.patches.Polygon = ax_inset.add_patch(matplotlib.patches.Polygon(bbox_ego.cpu().numpy().tolist(), fill=None, edgecolor=egoplotline.get_color()))
bbox_target_patch : matplotlib.patches.Polygon = ax_inset.add_patch(matplotlib.patches.Polygon(bbox_target.cpu().numpy().tolist(), fill=None, edgecolor=targetplotline.get_color()))

gaussian_pdf_vals_visible = gaussian_pdf_vals[:,visible_index]
glpoints_pdfratios = torch.max(gaussian_pdf_vals_visible, dim=1)[0]/torch.max(gaussian_pdf_vals_visible)
glpoints_sizes = (2**5.5)*(glpoints_pdfratios**0.75)
glpoints_scatter = ax_inset.scatter(*(gauss_pts[:,visible_index].T.cpu()), color=egoplotline.get_color(), s=glpoints_sizes.cpu(), zorder=3)
# ax_inset.scatter(*(gaussianmeans_target[visible_index].T.cpu()), color=targetplotline.get_color(), s=(2**4.5), zorder=3)
all_points = torch.cat([gauss_pts[:,visible_index], gaussianmeans_target[visible_index], bbox_ego, bbox_target], dim=0)


ax_inset.set_xlim(all_points[:,0].min().item() - .25*image_scale*car_length, all_points[:,0].max().item() + .25*image_scale*car_length)
ax_inset.set_ylim(all_points[:,1].min().item() - .25*image_scale*car_length, all_points[:,1].max().item() + .25*image_scale*car_length)
ax_inset.set_aspect(aspect=1.0, adjustable="box")
ax_inset.axis('off')
fig_inset.savefig(os.path.join(plotsdir, "%s.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)





In [ ]:
import matplotlib.patches, matplotlib.ticker
figname="Collision Prob Vs Time"
plt.close(fig=figname)
usetex = True
plt.rc("text", usetex=usetex)
figsize = np.asarray([columnwidth, columnwidth])
fig_time, ax_time = plt.subplots(label=figname, figsize=figsize, frameon=False)
no_collision_probs = torch.prod(1.0 - dense_collision_probs, dim=-1)
overall_collision_probs = 1.0 - no_collision_probs
alpha = 0.5
lambta_t = alpha*overall_collision_probs + (1.0-alpha)*(overall_collision_probs/(1.0-overall_collision_probs))
lambda_t_plot, = ax_time.plot(timage.cpu(), overall_collision_probs.cpu(), label="$P(t)$", linestyle="--", color="black")
lambda_t_plot, = ax_time.plot(timage.cpu(), lambta_t.cpu(), label="$\\lambda(t)$", color="black")

probinset = overall_collision_probs[visible_index].item()
fake_poly = ax_time.add_patch(matplotlib.patches.Rectangle((0,0),50,50, edgecolor=None, facecolor="grey", label="$m(T_F)$"))
fig_legend, ax_legend, bbox_legend = utils.export_legend(ax_time, fontsize=11)
fig_legend.savefig(os.path.join(plotsdir, "%s.legend.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches=bbox_legend)
plt.close(fig=fig_legend)
fake_poly.remove()
ax_time.relim()
ax_time.fill_between(timage.cpu(), 0.0, lambta_t.cpu(), alpha=0.5, color=fake_poly.get_facecolor())
max_lambda_idx = lambta_t.argmax().item()
t1 = 1.35#timage[max_lambda_idx].item()
t2 = 4.5
step=0.75
ticks = np.sort(np.asarray([0.0, 2.5, 3.5, 6.0, t1, t2]))
# ticks = np.linspace()

def tickformat(tick_value, tick_number):
    # if tick_value in [t1, t2]:
    #     return "%1.2f" % (tick_value) #
    if tick_value==t1:
        return  "%1.2f" % (tick_value)
    if tick_number==ticks.shape[0]-1:
        return "$T_F=%s$" % (tick_value,)
    return "%1.1f" % (tick_value) #
    #     return "$T_F$" if tick_value==t1 else "$T_F+\\Delta T$"
    # else:
    #     return "$%.1f$" % (tick_value)
ax_time.tick_params(axis="both", which="both", labelsize=(8*figsize[0]/columnwidth) if usetex else 8 )
ax_time.xaxis.set_ticks(ticks)
ax_time.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(tickformat))
ticklines=ax_time.xaxis.get_ticklines()
ticklabels=ax_time.xaxis.get_majorticklabels()
ticklocs=ax_time.xaxis.get_majorticklocs()
for (i, ticklock) in enumerate(ticklocs):
    if ticklock==t1:
        ticklabels[i].set_color("red")
        ticklines[i].set_color("red")
    if ticklock==t2:
        ticklabels[i].set_color("green")
        ticklines[i].set_color("green")
# tinset = timage[visible_index].item()
# tinset = [1.5, 4,5]
# ax_time.vlines(tinset, 0.0, probinset, color="black", linestyle="dashed")
ax_time.yaxis.tick_right()
ax_time.set_xlabel("Time (s)", fontsize=8*figsize[0]/columnwidth, labelpad=1.25*figsize[0]/columnwidth)
# ax_time.set_ylabel("$\\lambda(t)$", fontsize=8*figsize[0]/columnwidth)
fig_time.savefig(os.path.join(plotsdir, "%s.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)

In [ ]:
import deepracing_models.math_utils.collision_checking as cc
import matplotlib.collections, matplotlib.patches
tplot = dT_desired*torch.linspace(0.25, 0.9, steps=100).type_as(Curveparticles_mean)
(ego_pplot,), _ = mu.compositeBezierEval(Curveparticle_tstart[[0,]], Curveparticle_dT[[0,]], Curveparticles_mean, tplot[None])
(ego_vplot,), _ = mu.compositeBezierEval(Curveparticle_tstart[[0,]], Curveparticle_dT[[0,]], Curveparticles_mean_deriv, tplot[None])
tau0 = ego_vplot[0]/torch.norm(ego_vplot[0], p=2.0, keepdim=False)
nu0 = tau0[[1,0]]*flip
R0 = torch.stack([tau0, nu0], dim=1)
R0inv = R0.T
P0inv = -(R0inv@ego_pplot[0,:,None])[...,0]
(target_pplot,), _ = mu.compositeBezierEval(Targetvehicle_tstart[None], Targetvehicle_dT[None], Targetvehicle_curve, tplot[None])

ego_pfit = (R0inv@ego_pplot.T).T + P0inv
target_pfit = (R0inv@target_pplot.T).T + P0inv

target_pfit[:,0]+=0.5
target_pfit[:,1]-=1.5

ib_plot_local = (R0inv@ib_plot.T).T + P0inv
ob_plot_local = (R0inv@ob_plot.T).T + P0inv

kbezier=7
smc = torch.linspace(0.0, 1.0, steps=100).type_as(ego_pfit)
Mplot = mu.bezierM(smc[None], kbezier)[0]
Mderiv = mu.bezierM(smc[None], kbezier-1)[0]
_, (attacker_curve,) = mu.bezierLsqfit(ego_pfit[None], kbezier, t=tplot[None])
attacker_curve_plot = Mplot@attacker_curve
attacker_curve_deriv = kbezier*torch.diff(attacker_curve, dim=-2)/(tplot[-1]-tplot[0])
attacker_vels =  Mderiv@attacker_curve_deriv
attacker_tau = attacker_vels/torch.norm(attacker_vels, dim=-1, p=2.0, keepdim=True)
attacker_nu = attacker_tau[:,[1,0]]*flip
attacker_R = torch.stack([attacker_tau, attacker_nu], dim=-1)
attacker_boxpoints = torch.matmul(attacker_R, boxpoints).transpose(-2,-1) + attacker_curve_plot[:,None]
attacker_anchorpoints = attacker_boxpoints[...,0,:]

_, (defender_curve,) = mu.bezierLsqfit(target_pfit[None], kbezier, t=tplot[None])
defender_curve[...,1]+=1.5
defender_curve_plot = Mplot@defender_curve
defender_curve_deriv = kbezier*torch.diff(defender_curve, dim=-2)/(tplot[-1]-tplot[0])


curve_covars = torch.as_tensor([[[ 1.19892773e-02,  9.36912990e-08],
                                    [ 9.36912990e-08,  1.19892773e-02]],
                                [[ 2.02386101e-01, -3.81924440e-06],
                                    [-3.81924440e-06,  2.02386101e-01]],
                                [[ 4.34330645e-01,  2.91923870e-05],
                                    [ 2.91923870e-05,  4.34330645e-01]],
                                [[ 1.14617000e+00, -9.01698636e-05],
                                    [-9.01698636e-05,  1.14617000e+00]],
                                [[ 1.94844307e+00,  1.42177334e-04],
                                    [ 1.42177334e-04,  1.94844307e+00]],
                                [[ 1.90664060e+00, -1.19909455e-04],
                                    [-1.19909455e-04,  1.90664060e+00]],
                                [[ 3.54434947e+00,  4.76160528e-05],
                                    [ 4.76160528e-05,  3.54434947e+00]],
                                [[ 1.21235046e+00, -5.08694543e-06],
                                    [-5.08694543e-06,  1.21235046e+00]]]).type_as(ego_pplot)
curve_covars[:,0,1] = curve_covars[:,1,0] = 0.0
curve_covars_sparse = curve_covars.clone()
curve_covars_sparse[1:]*=5.0

figname="Example Curves Sparse"
plt.close(fig=figname)
fig_sparse, ax_sparse = plt.subplots(1,1, figsize=figsize, label=figname, layout="constrained", frameon=False)
ax_sparse.set_axis_off()
attacker_line, = ax_sparse.plot(*(attacker_curve_plot.cpu().T), color=utils.COLORS.UVA_ORANGE, label="Tego", linewidth=1.25)
defender_line, = ax_sparse.plot(*(defender_curve_plot.cpu().T), color=1.0-utils.COLORS.UVA_ORANGE, label="p(T)", linewidth=attacker_line.get_linewidth(), zorder=attacker_line.get_zorder())
fig_sparse.savefig(os.path.join(plotsdir, "%s.bare.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)
targetcurve_dist = torch.distributions.MultivariateNormal(defender_curve, covariance_matrix=curve_covars_sparse)
sampled_curves = targetcurve_dist.sample([2**12,])
sampled_curve_derivs = kbezier*torch.diff(sampled_curves, dim=-2)/(tplot[-1]-tplot[0])
sampled_curve_plot = Mplot@sampled_curves
defender_vels =  Mderiv@sampled_curve_derivs
defender_tau = defender_vels/torch.norm(defender_vels, dim=-1, p=2.0, keepdim=True)
defender_angles = torch.atan2(defender_tau[...,1], defender_tau[...,0])*(180.0/np.pi)
defender_nu = defender_tau[...,[1,0]]*flip
defender_R = torch.stack([defender_tau, defender_nu], dim=-1)
defender_boxpoints = torch.matmul(defender_R, boxpoints).transpose(-2,-1) + sampled_curve_plot[...,None,:]
defender_anchorpoints = defender_boxpoints[...,0,:]
res_dense = cc.rectangle_intersections2(attacker_boxpoints[None].expand_as(defender_boxpoints), defender_boxpoints, check_singular=True)
Pcol_t_rect : torch.Tensor = res_dense.collision_idx.sum(dim=0)/res_dense.collision_idx.shape[0]
Pcol_t_rect_argmax = Pcol_t_rect.argmax().item()
Pcol_t_rect_max = Pcol_t_rect[Pcol_t_rect_argmax].item()
any_collision_idx = res_dense.collision_idx.any(dim=-1)
ax_sparse.set_aspect(1.0)


car_diameter = float(np.sqrt(car_width**2 + car_length**2))
deltas_plot = sampled_curve_plot - attacker_curve_plot[None]
deltas_norms_plot = torch.norm(deltas_plot, p=2.0, dim=-1)
collision_idx_circ = deltas_norms_plot<car_diameter
Pcol_t_circ = collision_idx_circ.sum(dim=0)/collision_idx_circ.shape[0]
Pcol_t_circ_argmax = Pcol_t_circ.argmax().item()
Pcol_t_circ_max = Pcol_t_circ[Pcol_t_circ_argmax].item()

fig_sparse.savefig(os.path.join(plotsdir, "%s.bare.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)

s1 = ax_sparse.scatter(*attacker_curve_plot[Pcol_t_rect_argmax], s=2**1.5, color=attacker_line.get_color())
s2 = ax_sparse.scatter(*defender_curve_plot[Pcol_t_rect_argmax], s=2**1.5, color=defender_line.get_color())
fig_sparse.savefig(os.path.join(plotsdir, "example_curves.rectmarked.svg"), transparent=True, bbox_inches="tight", pad_inches=0.0)
s1.remove()
s2.remove()

s1 = ax_sparse.scatter(*attacker_curve_plot[Pcol_t_circ_argmax], s=2**1.5, color=attacker_line.get_color())
s2 = ax_sparse.scatter(*defender_curve_plot[Pcol_t_circ_argmax], s=2**1.5, color=defender_line.get_color())
fig_sparse.savefig(os.path.join(plotsdir, "example_curves.circlemarked.svg"), transparent=True, bbox_inches="tight", pad_inches=0.0)
s1.remove()
s2.remove()

figsize_pt=0.85*columnwidth*np.ones(2, dtype=np.float64)

# label="Pcol Circ"
# plt.close(fig=label)
# fig_circ, ax_circ = plt.subplots(1,1, label=label, frameon=False, layout="constrained", figsize=figsize_pt)
# ax_circ.plot(tplot, Pcol_t_circ, linewidth=1, color="black")
# ax_circ.set_xticks(tplot[[0,-1]].cpu(), labels=["", ""])
# ax_circ.set_ylim([0, Pcol_t_circ_max+0.05])
# ax_circ.yaxis.set_tick_params(labelsize=8*columnwidth/figsize_pt[0])
# fig_circ.savefig(os.path.join(plotsdir, "%s.svg" % (label.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)

# label="Pcol Rect"
# plt.close(fig=label)
# fig_rect, ax_rect = plt.subplots(1,1, label=label, frameon=False, layout="constrained", figsize=figsize_pt)
# ax_rect.plot(tplot, Pcol_t_rect, linewidth=1, color="black")
# ax_rect.set_xticks(tplot[[0,-1]].cpu(), labels=["", ""])
# ax_rect.set_ylim([0, Pcol_t_circ_max+0.05])
# ax_rect.yaxis.set_tick_params(labelsize=8*columnwidth/figsize_pt[0])
# fig_rect.savefig(os.path.join(plotsdir, "%s.svg" % (label.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)


# print(defender_anchorpoints.shape)
# print(defender_angles.shape)
# print(attacker_anchorpoints.shape)
# print(attacker_angles.shape)
step=int(round(attacker_boxpoints.shape[0]/20))
for j in range(0,attacker_boxpoints.shape[0],step):
    patch = ax_sparse.add_patch(matplotlib.patches.Polygon(attacker_boxpoints[j].cpu().numpy(), edgecolor=attacker_line.get_color(), fill=False))
# color=defender_line.get_color()
for i in range(8):
    curve_collides = any_collision_idx[i].item()
    patches = []
    #patch_collection : matplotlib.collections.PatchCollection
    for j in range(0,defender_boxpoints.shape[1],step):
        # rectangle_collides = res_dense.collision_idx[i,j]
        color = "red" if curve_collides else "green"
        # color = "red" if curve_collides else "green"
        patch = ax_sparse.add_patch(matplotlib.patches.Polygon(defender_boxpoints[i,j].cpu().numpy(), edgecolor=color, fill=False, alpha=0.3))
# for j in range(sampled_curve_plot.shape[0]):
#     color = "red" if any_collision_idx[j] else "green"
#     sparse_curve_lines.append(ax_sparse.plot(*(sampled_curve_plot[j].cpu().T), color=color, alpha=0.2, linestyle="dashed", zorder=attacker_line.get_zorder()-1)[0])
fig_sparse.savefig(os.path.join(plotsdir, "%s.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)
# for curveline in sparse_curve_lines:
#     curveline.remove()

figname="Example Curves"
plt.close(fig=figname)
fig_example, ax_example = plt.subplots(1,1, figsize=figsize, label=figname, layout="constrained", frameon=False)
attacker_line, = ax_example.plot(*(attacker_curve_plot.cpu().T), color=utils.COLORS.UVA_ORANGE, label="Tego", linewidth=1.25)
defender_line, = ax_example.plot(*(defender_curve_plot.cpu().T), color=1.0-utils.COLORS.UVA_ORANGE, label="p(T)", linewidth=attacker_line.get_linewidth(), zorder=attacker_line.get_zorder())

ibplotline, = ax_example.plot(*(ib_plot_local.cpu().T), color="black", label="Track Boundaries")
ibplotline.set_zorder(egoplotline.get_zorder())
obplotline, = ax_example.plot(*(ob_plot_local.cpu().T), color=ibplotline.get_color(), linestyle=ibplotline.get_linestyle())
obplotline.set_zorder(egoplotline.get_zorder())


targetcurve_dist = torch.distributions.MultivariateNormal(defender_curve, covariance_matrix=curve_covars)
sampled_curves = targetcurve_dist.sample([200,])
sampled_curve_plot = Mplot@sampled_curves
for j in range(sampled_curve_plot.shape[0]):
    curve_line, = ax_example.plot(*(sampled_curve_plot[j].cpu().T), color=defender_line.get_color(), alpha=0.075)
ratios = np.linspace(0.01, 2.0,  num=20) 
alphas = np.linspace(1.0,  0.01, num=ratios.shape[0])**1.0
for j in range(curve_covars.shape[0]):
    center = defender_curve[j].cpu().numpy()
    stdev = curve_covars[j,0,0].sqrt().item()
    for k in range(ratios.shape[0]):
        circle : matplotlib.patches.Circle = ax_example.add_patch(matplotlib.patches.Circle(center, radius=ratios[k]*stdev, fill=False, edgecolor=defender_line.get_color(), alpha=alphas[k]))
        circle.set_linewidth(0.5)
ax_example.set_aspect(aspect=1.0, adjustable="datalim")
fig_example.savefig(os.path.join(plotsdir, "%s.svg" % (figname.lower().replace(" ","_"),)), transparent=True, bbox_inches="tight", pad_inches=0.0)


In [ ]:
import matplotlib.transforms, matplotlib.patches
figname="Circle Overlap"
plt.close(fig=figname)
fig_circle, ax_circle = plt.subplots(1,1, figsize=figsize, label=figname, layout="constrained", frameon=False)
car_diameter = float(np.linalg.norm(np.asarray([car_length, car_width]), ord=2.0, axis=0))
print(car_diameter)
affinemat = np.eye(3)
transformimage = matplotlib.transforms.Affine2D(matrix=affinemat) + ax_circle.transData
circle1 : matplotlib.patches.Circle = ax_circle.add_patch(
    matplotlib.patches.Circle([car_length/2.0, car_width/2.0], radius=car_diameter/2.0, fill=False, edgecolor=utils.COLORS.UVA_ORANGE, alpha=1.0, transform=transformimage))
# ax_circle.imshow()
ax_circle.set_xlim(-car_length, car_length)
ax_circle.set_ylim(-car_length, car_length)

